# Panorama Image Stitching

**CSC14120 Lập trình song song · Nhóm 16 · Báo cáo đồ án cuối kỳ**

Báo cáo đi theo [cấu trúc notebook của đề bài](docs/specs/CSC14120_2026_Final%20Project.md#52-notebook-report-structure). Để chạy bản nộp, mở notebook trong một phiên Google Colab mới có GPU Tesla T4 rồi chọn **Runtime → Run all**. Notebook có hai loại ô code. Loại thứ nhất đọc lại bằng chứng từ các báo cáo đã nghiệm thu ở từng giai đoạn. Loại thứ hai build chương trình và đo lại mọi phiên bản trong cùng phiên Colab.

### Mục lục

1. [Thông tin nhóm](#team-information)
2. [Giới thiệu](#introduction)
3. [Phương pháp đã chọn](#chosen-methods)
4. [V0 tuần tự](#v0-sequential)
5. [V1 CUDA cơ bản](#v1-basic-cuda)
6. [Các phiên bản tối ưu](#optimized-versions)
7. [So sánh tổng hợp](#final-comparison)
8. [Kết quả ghép ảnh](#results-gallery)
9. [Kết luận](#conclusion)
10. [Tài liệu tham khảo](#references)

<a id="team-information"></a>

## 1. Thông tin nhóm

| Thành viên | MSSV | V0 và V1 | Các phiên bản tối ưu và sản phẩm cuối kỳ |
| --- | --- | --- | --- |
| Nguyễn Hồ Anh Tuấn (nhóm trưởng) | 23120185 | S1 tiền xử lý | Kernel S5 của V2 và V3, nghiệm thu tích hợp V4, slide và tư liệu thuyết trình |
| Trần Hữu Kim Thành | 23120166 | S2 phát hiện và S3 mô tả | Tích hợp pipeline và CLI cho V2/V3/V4, chạy nghiệm thu V3, phần dữ liệu và trực quan hóa của notebook cuối kỳ |
| Lê Thượng Đế | 23120232 | S4 so khớp và S5 homography | Benchmark V2/V3/V4, profiling Nsight và APOD, slide và tư liệu thuyết trình |
| Lê Hà Thanh Chương | 23120195 | S6 warp và blend | Hạ tầng kiểm thử V2/V3, kernel S6 V4 và test, cấu trúc notebook cuối kỳ và phân tích kỹ thuật |

[Phân công stage ban đầu](docs/report/3.1-team-plan.md) · [Phân công và bàn giao theo giai đoạn](docs/task-board.md)

### Môi trường chạy và phiên bản source

Nếu notebook được mở từ một bản checkout của repo, ô setup dùng luôn bản đó. Nếu không, ô này tải bản source đã ghim bằng `SOURCE_REF`. Phần đo thời gian cần Tesla T4, CUDA toolkit và script build của repo. Các ô đọc bằng chứng lịch sử chạy được cả trên kernel Python chỉ có CPU. Bản source đang ghim đã có đủ V0–V5. Trước lượt chạy nộp bài, cần ghim lại `SOURCE_REF` vào SHA của bản tích hợp cuối.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/nh-atuan/panorama-stitching.git"
SOURCE_REF = "696fc82c72b2098ea7ca24c7c89efdab9d3452ec"
WORK_DIR = Path.cwd()
REPO_ROOT = WORK_DIR if (WORK_DIR / "scripts/build.py").is_file() else WORK_DIR / "panorama-stitching"

# Fetch the report sources only when a checkout is unavailable.
if not (REPO_ROOT / "scripts/build.py").is_file():
    if REPO_ROOT.exists():
        raise RuntimeError(f"Expected a repository checkout or a new directory at {REPO_ROOT}.")
    subprocess.run(["git", "init", str(REPO_ROOT)], check=True)
    subprocess.run(["git", "fetch", "--depth", "1", REPO_URL, SOURCE_REF], cwd=REPO_ROOT, check=True)
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_ROOT, check=True)

DATASET_DIR = Path(os.environ.get("GROUP16_DATASET_DIR", str(REPO_ROOT / "dataset"))).resolve()
BENCHMARK_DIR = Path(
    os.environ.get("GROUP16_BENCHMARK_DIR", str(REPO_ROOT / "results" / "final"))
).resolve()
BASELINE_VERSIONS = (0, 1, 2, 3, 4, 5)
TIMED_RUNS = 5

source = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True, check=False
)
print(f"Source checkout: {source.stdout.strip() or 'local source package (no Git metadata)'}")
print(f"Timed runs per configuration: {TIMED_RUNS}")

for tool, args in (
    ("nvidia-smi", ["--query-gpu=name,driver_version", "--format=csv,noheader"]),
    ("nvcc", ["--version"]),
):
    executable = shutil.which(tool)
    if executable is None:
        print(f"{tool} unavailable in this runtime.")
        continue
    result = subprocess.run([executable, *args], capture_output=True, text=True, check=False)
    print(result.stdout.strip() if result.returncode == 0 else f"{tool} unavailable: {result.stderr.strip()}")


### Dữ liệu và lượt benchmark chung

Ô dưới tải bộ dataset cuối kỳ từ [thư mục Drive](https://drive.google.com/drive/folders/1_J_q5q-K1g7ZNnczTWvXBzwpOsof9ghL) (xem [hướng dẫn dataset](docs/dataset.md)), kiểm tra SHA-256 của `manifest.csv` và của từng ảnh, rồi build `stitch` bằng `scripts/build.py`. Tiếp theo, ô này hỏi chính file `stitch` vừa build xem nó nhận những phiên bản nào và chạy `scripts/run_benchmark.py` một lần cho tất cả các phiên bản đó, có cả cặp challenge. Kết quả được ghi vào `BENCHMARK_DIR`.

Các bảng và biểu đồ ở mục 4 đến mục 8 đều đọc lại từ thư mục này. Nhờ vậy mọi phiên bản được đo trên cùng một GPU, trong cùng một phiên Colab. Code của các bước nằm trong [`scripts/notebook/final_notebook.py`](scripts/notebook/final_notebook.py).

In [ ]:
import sys

sys.path.insert(0, str(REPO_ROOT / "scripts"))
from notebook import final_notebook as fn
from IPython.display import Image, Markdown, display

DATASET_DIR, verified_images = fn.fetch_dataset(DATASET_DIR, WORK_DIR / "dataset_drive")
print(f"Dataset: {DATASET_DIR} ({len(verified_images)} images match manifest.csv)")
STITCH = fn.build_stitch(REPO_ROOT)
run_versions = fn.supported_versions(STITCH)
missing_versions = sorted(set(BASELINE_VERSIONS) - set(run_versions))
if missing_versions:
    raise RuntimeError(f"This build does not accept versions {missing_versions}")
print("Versions in this build:", ", ".join(f"V{version}" for version in run_versions))

if (BENCHMARK_DIR / "environment.json").is_file():
    print(f"Reusing the benchmark session in {BENCHMARK_DIR}")
else:
    fn.run_benchmark_session(REPO_ROOT, STITCH, DATASET_DIR, BENCHMARK_DIR, run_versions, TIMED_RUNS)

benchmark = fn.load_benchmark(BENCHMARK_DIR, run_versions)
benchmark_summary = benchmark.summary
benchmark_speedup = benchmark.speedup
benchmark_failures = benchmark.failures
benchmark_environment = benchmark.environment
CHART_DIR = BENCHMARK_DIR.with_name(BENCHMARK_DIR.name + "_charts")
print(f"GPU: {benchmark_environment['gpu']} | measured at {benchmark_environment['time_utc']} | "
      f"timed runs per configuration: {benchmark_environment['runs']}")

<a id="introduction"></a>

## 2. Giới thiệu

Bài toán là ghép hai ảnh RGB có vùng chồng lấn thành một panorama. Ảnh A là hệ quy chiếu. Homography $H$ ánh xạ điểm của ảnh B sang A, và canvas đầu ra bao trọn A cùng vùng ảnh B sau khi biến đổi. Một homography duy nhất hợp nhất với cảnh gần phẳng, hoặc khi camera chủ yếu xoay và ít tịnh tiến. Cảnh có nhiều lớp độ sâu hay có vật chuyển động có thể để lại chỗ lệch sau khi ghép.

```mermaid
flowchart LR
    A["Ảnh A"] --> A1["S1 tiền xử lý"] --> A2["S2 phát hiện"] --> A3["S3 mô tả"]
    B["Ảnh B"] --> B1["S1 tiền xử lý"] --> B2["S2 phát hiện"] --> B3["S3 mô tả"]
    A3 --> M["S4 so khớp"]
    B3 --> M
    M --> H["S5 ước lượng H"] --> W["S6 warp và blend"] --> P["Panorama"]
    A -. RGB .-> W
    B -. RGB .-> W
```

Hai nhánh đặc trưng không phụ thuộc nhau cho tới bước so khớp. S6 nhận homography đã ước lượng cùng hai ảnh RGB gốc. Nhóm đánh giá riêng ba mặt: hình học, độ trung thực của panorama và thời gian chạy. Hai phiên bản cho ra cùng pixel trên một ca kiểm thử chỉ chứng tỏ hai bản cài đặt tương đương nhau. Chất lượng căn chỉnh vẫn phụ thuộc vào homography ước lượng được và vào cảnh chụp.

<a id="chosen-methods"></a>

## 3. Phương pháp đã chọn

| Stage | Phương pháp | Lý do và giới hạn |
| --- | --- | --- |
| S1 · Tiền xử lý | Chuyển RGB xen kẽ sang ảnh xám chuẩn hóa theo hệ số luma BT.601, sau đó làm mượt bằng Gaussian tách được, biên ảnh xử lý bằng clamp. | Làm mượt giảm nhiễu trước khi phát hiện góc. Lượt ngang và lượt dọc dùng stencil riêng cho từng pixel, không phụ thuộc nhau. [Ghi chú S1](docs/report/3.3-s1-preprocessing.md) |
| S2 · Phát hiện | Phép thử đoạn FAST, non-maximum suppression và chọn top-K tất định. | Phép thử góc chỉ dùng lân cận của pixel nên hợp với cách chạy mỗi pixel một thread. NMS bỏ bớt các góc nằm sát nhau, còn giới hạn K chặn chi phí so khớp ở phía sau. [Rosten và Drummond](https://www.edwardrosten.com/work/rosten_2006_machine.pdf) · [Ghi chú S2](docs/report/3.3-s2-detection.md) |
| S3 · Mô tả | BRIEF không xoay trên một ảnh xám được làm mượt riêng, dùng mẫu cố định do nhóm sinh và kiểm tra cửa sổ lấy mẫu. | Các phép so sánh cường độ nhị phân cho descriptor gọn và so khớp nhanh. Vì không ước lượng hướng, descriptor kém bền khi ảnh xoay nhiều trong mặt phẳng. [Calonder và cộng sự](https://www.cs.ubc.ca/~lowe/525/papers/calonder_eccv10.pdf) · [Ghi chú S3](docs/report/3.3-s3-description.md) |
| S4 · So khớp | Tìm vét cạn theo khoảng cách Hamming từ A sang B, giữ hai khoảng cách nhỏ nhất rồi áp ratio test. Cấu hình production tắt cross-check. | Ratio test loại các cặp có hai ứng viên gần bằng nhau. Mỗi descriptor của A được xử lý độc lập, khối lượng việc tỉ lệ với tích số descriptor của hai ảnh. [Lowe](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf) · [Ghi chú S4](docs/report/3.3-s4-matching.md) |
| S5 · Homography | RANSAC có seed với DLT chuẩn hóa trên mẫu tối thiểu, chấm điểm MSAC theo sai số chiếu thuận B→A, điều kiện dừng tất định và fit lại trên các inlier. | Ước lượng bền vững loại được phần lớn cặp khớp sai. Ngưỡng inlier tỉ lệ với đường chéo ảnh A. Cấu trúc lặp và mẫu suy biến vẫn có thể cho ra mô hình không dùng được. [Brown và Lowe](https://www.cs.ubc.ca/~lowe/papers/07brown.pdf) · [Ghi chú S5](docs/report/3.3-s5-homography.md) |
| S6 · Warp và blend | Dựng canvas có giới hạn kích thước, ánh xạ ngược từng pixel canvas về B bằng $H^{-1}$, lấy mẫu B bằng nội suy bilinear tự cài, và lấy trung bình RGB với trọng số bằng nhau ở pixel cả hai ảnh cùng phủ. | Ánh xạ ngược không để lại lỗ như cách đẩy pixel xuôi. Coverage xác định theo hình học, nên pixel màu đen nằm trong ảnh vẫn được tính. Pixel nằm ngoài cả hai ảnh có màu đen. [Szeliski](https://szeliski.org/Book/) · [Ghi chú S6](docs/report/3.3-s6-warp-blend.md) |

Với phép chiếu về tọa độ thường $\pi(x,y,w)=(x/w,y/w)$, ánh xạ thuận là $p_A=\pi(H\tilde p_B)$. Ở S6, offset của canvas được trừ đi trước khi nhân với ma trận nghịch đảo. Bước lấy mẫu kiểm tra tọa độ theo biên ảnh nguồn và clamp các lân cận bilinear ở hàng và cột cuối. RGB đầu ra được lượng tử hóa sau khi nội suy và blend.

In [ ]:
def show_source(relative, identifiers):
    """Print the selected production statements and identify their source."""
    lines = (REPO_ROOT / relative).read_text(encoding="utf-8").splitlines()
    print(f"\n{relative}")
    for item in identifiers:
        matches = [(number, line) for number, line in enumerate(lines, 1) if item in line]
        if not matches:
            raise ValueError(f"{relative}: missing source statement {item!r}")
        number, line = matches[0]
        print(f"{number:>4}  {line.strip()}")

show_source("src/common/v0_config.h", (
    "kV0GaussianSigma =", "kV0GaussianRadius =", "kV0FastThreshold =", "kV0MaxKeypoints =",
))
show_source("src/common/fast_corner.h", ("kFastArcLength =",))
show_source("src/common/brief_pattern.h", (
    "kBriefTests =", "kBriefPatchRadius =", "kBriefSmoothSigma =", "kBriefSmoothRadius =",
))
show_source("src/common/feature_support.h", (
    "kRatio =", "kMaxIterations =", "kConfidence =", "kSeed =", "kReferenceThreshold =",
))
show_source("src/common/feature_support.cpp", ("candidate.match_cross_check =", "std::hypot(1920.0"))
show_source("src/common/s6_cuda_setup.cuh", (
    "kCoordinateEpsilon =", "kMaxCanvasDimension =", "kMaxCanvasPixels =",
))

<a id="v0-sequential"></a>

## 4. V0 tuần tự

V0 chạy cả sáu stage trên một thread C++. Ở các stage đặc trưng, hai ảnh được xử lý lần lượt. Descriptor và match nằm trên host, S6 duyệt canvas theo từng hàng. CLI dựng cấu hình dùng chung từ ảnh A, giải mã cả hai ảnh trước khi bắt đầu đo và mã hóa panorama sau khi đo xong. V0 dùng seed lấy mẫu cố định, giữ thứ tự ổn định khi chọn đặc trưng và mô hình, và đo thời gian bằng `std::chrono::high_resolution_clock`.

V0 là mốc tham chiếu để đối chiếu hình học và RGB của các phiên bản khác. Lượt đo giữa kỳ cho thấy S4 tốn nhiều thời gian nhất ở small và medium. Ở large, stage lớn nhất chuyển sang S2 hoặc S6 tùy site. Khối lượng so khớp phụ thuộc vào số descriptor, còn thời gian dựng canvas tăng theo diện tích canvas và theo số mẫu hợp lệ lấy từ B. [Cài đặt V0](src/v0/v0_pipeline.cpp) · [Thời gian đo giữa kỳ](docs/report/3.5-timing.md)

In [ ]:
# One schema-checked table reader is shared by historical cells and the acceptance audit.
from math import exp, fsum, log
from v5_report import read_table

def report_table(relative, first_header):
    return read_table(REPO_ROOT / relative, first_header)

V0_REPORT = "docs/report/3.5-timing.md"
v0_timing = report_table(V0_REPORT, ("Site", "Kích thước", "S1", "S2"))
stage_columns = tuple(f"S{stage}" for stage in range(1, 7))
print(f"Historical V0 timing, ms · {V0_REPORT}")
print("Case                S1       S2       S3       S4       S5       S6    Total   Hot")
for row in v0_timing:
    case = f"{row['Site']}/{row['Kích thước']}"
    hot = max(stage_columns, key=lambda stage: float(row[stage]))
    times = " ".join(f"{float(row[col]):8.3f}" for col in (*stage_columns, "Total"))
    print(f"{case:15s} {times}   {hot}")

show_source("src/v0/v0_pipeline.cpp", ("v0_preprocess(", "v0_detect(", "v0_describe(",
                                      "v0_match(", "v0_homography(", "v0_warp_blend("))

### Thời gian V0 trong lượt benchmark chung

Bảng thứ nhất là thời gian trung bình của từng stage, tính bằng ms, cho mọi cấu hình của lượt benchmark ở mục 1. Cột cuối ghi stage chiếm nhiều thời gian nhất và tỉ lệ của nó trong `Total_ms`. Bảng thứ hai lấy trung bình tỉ lệ của mỗi stage trên các site chuẩn, tách theo độ phân giải. Ảnh cuối là panorama V0 của site1 ở độ phân giải small, sinh ra trong cùng lượt chạy.

In [ ]:
display(Markdown(fn.stage_table(benchmark, "V0")))
display(Markdown(fn.dominant_stage_table(benchmark, "V0")))
display(Image(data=fn.panorama_preview(benchmark, "V0", "site1", "small")))

<a id="v1-basic-cuda"></a>

## 5. V1 CUDA cơ bản

V1 giữ ảnh, buffer đặc trưng, match và homography trong workspace GPU suốt các stage. RGB đầu vào được chép lên device, các buffer ảnh và đặc trưng trung gian dùng chung giữa các lần gọi stage, và canvas hoàn chỉnh được chép về host. Metadata điều khiển, trong đó có kích thước canvas mà S6 cần để cấp phát, được đọc về riêng. Các lần đồng bộ để đọc metadata, overhead launch và các bước điều khiển tuần tự có thể làm giảm lợi ích của kernel song song.

| Stage | Cách V1 chia việc trên GPU |
| --- | --- |
| S1 | Lưới pixel hai chiều cho bước chuyển xám và cho từng lượt blur. |
| S2 | Mỗi thread chấm điểm FAST cho một pixel và loại các điểm không phải cực đại địa phương. Bước nén candidate và sort tất định cho ra top-K keypoint. |
| S3 | Lưới một chiều duyệt các keypoint candidate. Một phép scan gán vị trí đầu ra sau khi nén, rồi mỗi thread dựng descriptor BRIEF cho một keypoint với mẫu cố định đặt trong constant memory. |
| S4 | Mỗi block tìm trong B cho một descriptor của A bằng popcount Hamming. Phép reduction giữ hai candidate gần nhất, sau đó các bước prefix và scatter gom các match được nhận. |
| S5 | Các block tính và chấm điểm từng lô giả thuyết. Bước commit tuần tự giữ đúng prefix tất định và quy tắc dừng, còn bước refit của baseline chạy trên một thread. |
| S6 | Bước setup tính hình học canvas. Sau khi xóa canvas, một lưới pixel chép A, rồi một lưới canvas gather B và blend vùng overlap. |

Thời gian của các stage GPU và của các lần chép dữ liệu đo bằng CUDA event. `Total_ms` là khoảng end-to-end có đồng bộ của pipeline, gồm H2D, các stage, D2H và phần điều phối phía host nằm trong khoảng đó. `Total_Kernel_ms` là tổng thời gian kernel, đo tại từng lần launch. Thời gian một stage có thể gồm cả bước setup hoặc đồng bộ. Vì vậy tổng các stage và tổng kernel cộng thời gian chép dữ liệu là hai đại lượng khác nhau. Đọc và ghi file nằm ngoài vùng đo. Mỗi cấu hình có warm-up trước các lượt đo. [Pipeline V1](src/v1/v1_pipeline.cu) · [Interface đo thời gian](docs/interface.md) · [Hướng dẫn đo thời gian CUDA](https://docs.nvidia.com/cuda/archive/13.0.0/cuda-c-best-practices-guide/index.html#timing)

In [ ]:
P8_BENCHMARK = "docs/report/p8-t03-v3-apod.md"
P8_VERIFICATION = "docs/report/p8-v3-verification.md"
p8_totals = report_table(P8_BENCHMARK, ("Site", "Size", "V0 ms", "V1 ms"))
p8_real = report_table(P8_VERIFICATION, ("Cấu hình", "V0/V1/V2 tại P7"))

print("Historical P8 comparison, V0 and V1 measured in the same Colab Tesla T4 session")
print("Case                V0 ms     V1 ms   V0/V1")
for row in p8_totals:
    case = f"{row['Site']}/{row['Size']}"
    v0, v1 = float(row["V0 ms"]), float(row["V1 ms"])
    print(f"{case:15s} {v0:9.3f} {v1:9.3f} {v0 / v1:7.2f}x")

print(f"Real-image status parity across V0–V3 reported for {len(p8_real)} configurations.")
show_source("src/v1/v1_pipeline.cu", ("ErrorCode run_gpu_stages(",
                                      "v1_preprocess, v1_detect", "trace.stage_start[i].record"))

### Thời gian V1 trong lượt benchmark chung

Bảng đầu tách thời gian của V1 theo từng stage, kèm H2D, D2H và tổng thời gian kernel. Bảng sau so `Total_ms` và tổng kernel của V1 với V0 trên cùng cấu hình. Ô speedup để trống khi phiên bản dùng làm mốc không chạy được cấu hình đó. Kết quả đối chiếu V1 với V0 (homography, canvas và sai lệch RGB) được ghi trong [tài liệu tích hợp V1](docs/interface.md#118-pipeline-v1-tích-hợp-p6-t01).

In [ ]:
display(Markdown(fn.stage_table(benchmark, "V1")))
display(Markdown(fn.speedup_table(benchmark, "V1")))

<a id="optimized-versions"></a>

## 6. Các phiên bản tối ưu

Kết quả trong mục này lấy từ báo cáo của từng giai đoạn, mỗi báo cáo ghi rõ mốc so sánh của mình. Mục 7 dành cho lượt chạy tổng hợp của bản nộp. Với một điểm mẫu $q$ trong vùng overlap của B, sai số synthetic là $e(q)=\|\pi(Hq)-\pi(H_{gt}q)\|_2$. Suite kiểm tra RMSE và giá trị lớn nhất của các sai số này. Drift giữa hai phiên bản thì so phép chiếu của hai homography ước lượng được. Giá trị lớn nhất chỉ tính trên các điểm mẫu. Khi nghiệm thu, status được kiểm tra trước, sau đó tới homography, kích thước và offset canvas (phải khớp tuyệt đối), và RGB ở các ca thành công. [Ngưỡng nghiệm thu và mã hỗ trợ test](tests/gpu/pipeline_test_support.cuh)

### V2 · Song song hóa chấm điểm và refit của RANSAC

#### Bottleneck

Khi đo toàn pipeline ở chế độ production, S5 của V1 lớn hơn mọi stage khác. Việc dựng giả thuyết, chấm điểm và refit trên inlier kéo chậm cả pipeline, dù các kernel ảnh và descriptor phía trước đã chạy trên GPU. [Báo cáo APOD P7](docs/report/p7-t03-v2-apod.md) giải thích vì sao chỉ đo riêng từng kernel bằng profiler thì chưa đủ để kết luận thứ hạng này.

#### Ý tưởng

Chia việc đánh giá giả thuyết và refit trên toàn bộ inlier cho nhiều thread. Thứ tự commit giả thuyết vẫn giữ nguyên, để phần tính song song làm trước không làm đổi prefix của RANSAC.

#### Cài đặt

V2 thay lời gọi S5 và dùng lại V1 cho S1–S4 và S6. Các thread cùng làm các phép Jacobi SVD và chấm điểm MSAC. Kernel refit xử lý song song phần đóng góp của các inlier và dùng chung dữ liệu làm việc. Seed, DLT chuẩn hóa, ngưỡng sai số chiếu và cách chọn inlier giữ như cấu hình tham chiếu. [Source S5 V2](src/v2/s5/v2_homography.cu) · [Pipeline V2](src/v2/v2_pipeline.cu)

#### Kết quả

P7 so V2 với V1 trong cùng một phiên T4, còn tỉ số V2/V0 của P7 dùng baseline CPU đo từ trước. Đến P8, nhóm chạy lại V0 và V2 cùng nhau, nên so sánh end-to-end V2/V0 theo P8 được trình bày riêng ở dưới. Lượt nghiệm thu xác nhận các ngưỡng synthetic, trạng thái khớp nhau trên ảnh thật và RGB trùng khớp hoàn toàn ở các panorama thành công. [Nghiệm thu P7](docs/report/p7-v2-verification.md)

#### Phân tích

Pipeline nhanh lên nhiều nhất ở những ca trước đó bị S5 chi phối. Trong P7, S5 vẫn là stage lớn nhất ở các ca chuẩn, nhưng tại site2/large S6 đã gần bằng S5. Trace Nsight của V2 cho thấy kernel giả thuyết là chỗ nên tối ưu tiếp ở V3. Áp lực thanh ghi và occupancy đạt được thấp là lý do V3 đổi cách chia việc và cách reduction.

In [ ]:
def geomean(values):
    values = list(values)
    return exp(fsum(log(value) for value in values) / len(values))

P7_BENCHMARK = "docs/report/p7-t03-v2-apod.md"
P7_VERIFICATION = "docs/report/p7-v2-verification.md"
p7_totals = report_table(P7_BENCHMARK, ("Site", "Size", "V1 P6 total", "V1 P6 S5"))
p7_synthetic = report_table(P7_VERIFICATION, ("Fixture", "RMSE V2 (px)", "Max V2 (px)"))
p7_real = report_table(P7_VERIFICATION, ("Cấu hình", "Trạng thái chung", "Canvas"))
v2_over_v1 = [float(r["V1 P7 total"]) / float(r["V2 total"]) for r in p7_totals]
v2_over_v0_p8 = [float(r["V0 ms"]) / float(r["V2 ms"]) for r in p8_totals]
p7_success = [r for r in p7_real if r["Trạng thái chung"] == "Success"]
print("V2/V1 end-to-end speedup, same-session P7 Tesla T4:"
      f" {min(v2_over_v1):.2f}–{max(v2_over_v1):.2f}x, geometric mean {geomean(v2_over_v1):.2f}x")
print("V2/V0 end-to-end speedup, independent same-session P8 repeat:"
      f" {min(v2_over_v0_p8):.2f}–{max(v2_over_v0_p8):.2f}x")
print(f"P7 verification: {len(p7_synthetic)} synthetic cases and {len(p7_real)} real status comparisons.")
print(f"Max RGB difference across {len(p7_success)} successful panoramas:"
      f" {max(int(r['Max RGB']) for r in p7_success)} levels/channel.")

### V3 · Lập lịch kernel RANSAC và reduction

#### Bottleneck

Profiling P7 xác định `compute_hypotheses` là kernel lớn nhất của V2 trong các trace site1. Áp lực thanh ghi giới hạn occupancy ở những lần launch được profile, còn việc launch lặp lại theo từng lô làm tăng overhead lập lịch.

#### Ý tưởng

Dùng warp shuffle cho reduction MSAC, chia sẻ scratch của DLT và SVD trong cả block, và xử lý lô lớn hơn. Mục tiêu là mỗi multiprocessor có nhiều việc chạy cùng lúc hơn và số lần launch ít đi, trong khi vẫn giữ thứ tự commit giả thuyết.

#### Cài đặt

V3 cập nhật S5 trong pipeline GPU dùng chung. Kernel giả thuyết đặt ma trận thiết kế và các vector kỳ dị phải trong mảng shared, cộng điểm bằng reduction mức warp và khai báo launch bounds. Kích thước lô mới làm giảm số lần launch kernel giả thuyết. S1–S4 và S6 dùng lại V1. [Source S5 V3](src/v3/s5/v3_homography.cu) · [Pipeline V3](src/v3/v3_pipeline.cu) · [CUDA Best Practices Guide](https://docs.nvidia.com/cuda/archive/13.0.0/cuda-c-best-practices-guide/index.html)

#### Kết quả

Trong benchmark P8 trên Tesla T4, đo cùng phiên, V3 nhanh hơn V2 ở 8/9 cấu hình chuẩn, speedup end-to-end trung bình nhân là 1,85×. Mẫu số là `Total_ms` trung bình của V3 ở từng cấu hình. Tỉ số so với V0 dùng thời gian V0 của chính lượt P8. Tỉ số riêng S5 dùng khoảng CUDA event của stage này. [Benchmark và profiling P8](docs/report/p8-t03-v3-apod.md#4-result)

Lượt nghiệm thu P8 kiểm tra riêng các ngưỡng synthetic và parity trên ảnh thật, đồng thời chạy Compute Sanitizer trên suite V3 và một ca CLI. Thời gian của các lần test và sanitizer không đưa vào so sánh benchmark. [Nghiệm thu P8](docs/report/p8-v3-verification.md)

#### Phân tích

V3 chậm hơn V2 một chút ở site2/medium, nơi thời gian S5 gần như không đổi. Tại site2/large, S6 lên tới 28,468 ms, chiếm 45,7% `Total_ms` của V3, và đây là lý do nhóm chọn hướng tối ưu của V4. Nsight ghi nhận số lần launch giả thuyết giảm ở site1 và occupancy đạt được tăng ở lần launch site1/medium được lấy mẫu. Mẫu stall do barrier vẫn còn nhiều. Thí nghiệm đánh giá cả gói thay đổi cùng lúc nên không tách được tác dụng của từng thay đổi.

In [ ]:
p8_stage = report_table(P8_BENCHMARK, ("Site", "Size", "S5 V2 ms", "S5 V3 ms"))
p8_synthetic = report_table(P8_VERIFICATION, ("Ca", "Điểm Q theo P7"))
p8_memory = report_table(P8_VERIFICATION, ("Target", "Exit", "Errors", "Leaked bytes"))
show_source("tests/gpu/pipeline_test_support.cuh", ("kGrid =", "kTruthRmsePx =",
              "kTruthMaxPx =", "kDriftPx =", "kRgbTolerance ="))

ratios_v2 = [float(r["V2 ms"]) / float(r["V3 ms"]) for r in p8_totals]
ratios_v0 = [float(r["V0 ms"]) / float(r["V3 ms"]) for r in p8_totals]
ratios_s5 = [float(r["S5 V2 ms"]) / float(r["S5 V3 ms"]) for r in p8_stage]
v3_wins = sum(ratio > 1 for ratio in ratios_v2)
print("Historical P8 benchmark · Colab Tesla T4")
print(f"V3/V2 end-to-end: {v3_wins}/{len(ratios_v2)} faster,"
      f" geometric mean {geomean(ratios_v2):.2f}x.")
print(f"V3/V0 end-to-end: geometric mean {geomean(ratios_v0):.2f}x.")
print(f"V3/V2 S5 stage: geometric mean {geomean(ratios_s5):.2f}x.")

target = ("site2", "large")
large_total = next(r for r in p8_totals if (r["Site"], r["Size"]) == target)
large_stage = next(r for r in p8_stage if (r["Site"], r["Size"]) == target)
s6_ms = float(large_stage["S6 V3 ms"])
print(f"site2/large: S6 {s6_ms:.3f} ms of {float(large_total['V3 ms']):.3f} ms total"
      f" ({s6_ms / float(large_total['V3 ms']):.1%}).")

synthetic_errors = [
    tuple(float(value.removesuffix(" px")) for value in
          row["V3 RMSE / Max / drift / canvas / RGB"].split(" / ")[:2])
    for row in p8_synthetic
]
print(f"P8 acceptance: {len(synthetic_errors)} synthetic cases,"
      f" worst RMSE {max(rmse for rmse, _ in synthetic_errors):.3g} px,"
      f" worst point error {max(error for _, error in synthetic_errors):.3g} px.")
p8_real_status = "Trạng thái V0–V3, drift, canvas và RGB của lượt P8"
real_success = [row[p8_real_status] for row in p8_real if row[p8_real_status].startswith("Success")]
max_rgb = max(int(status.rsplit(" ", 1)[1]) for status in real_success)
print(f"Status parity reported for {len(p8_real)} real configurations,"
      f" Max RGB {max_rgb} across {len(real_success)} successful panoramas.")
for row in p8_memory:
    print(f"Memcheck {row['Target']}: exit {row['Exit']}, errors {row['Errors']},"
          f" leaked bytes {row['Leaked bytes']}.")

### V4 · Dựng canvas S6 trong một lượt

#### Bottleneck

Ở site2/large, benchmark P8 cho thấy S6 thành stage lớn nhất sau khi S5 đã được tối ưu. Renderer của V3 chạy ba lượt: xóa canvas, chép A, rồi gather B để nội suy và blend. Ở vùng overlap, lượt cuối đọc lại màu A vừa ghi vào canvas. Đây là cơ sở để chọn hướng gộp S6 cho V4. [Phân tích P8](docs/report/p8-t03-v3-apod.md#5-analysis)

#### Ý tưởng

Mỗi thread tính màu cuối cùng cho một pixel canvas trong một lượt duy nhất. Thread xác định coverage theo hình học, đọc thẳng từ A khi pixel nằm trong A, chiếu ngược về B và nội suy bilinear. Gộp các lượt dựng canvas giúp giảm số thao tác GPU và các bước ghi trung gian lên canvas. Lưu lượng DRAM thực tế phải kiểm chứng bằng profiler.

#### Cài đặt

V4 dùng lại S1–S4 của V1 và S5 của V3. S6 giữ nguyên phần chuẩn hóa và nghịch đảo H, kiểm tra hình học, tính canvas, offset và cấp phát output. Kernel 2D dùng block 32×8 và giữ nguyên cách baseline tính bilinear bằng double, snap, clamp và làm tròn. Pixel chỉ thuộc A hoặc chỉ thuộc B lấy màu của ảnh tương ứng, pixel overlap lấy trung bình 50/50, vùng không ảnh nào phủ có màu nền đen. Pixel đen trong ảnh nguồn vẫn được tính là hợp lệ vì coverage xét theo tọa độ. Canvas nằm trên device cho tới bước D2H của pipeline. [Source S6 V4](src/v4/s6/v4_warp_blend.cu) · [Pipeline V4](src/v4/v4_pipeline.cu)

#### Kết quả

Lượt P9 chính thức của Đế đo V0–V4 trên cùng một Tesla T4, mỗi cấu hình có một warm-up và năm lượt đo. Trên chín cấu hình chuẩn, V4 nhanh hơn V3 ở 8/9 ca, speedup end-to-end trung bình nhân là 1,0086×. So với V0 cùng phiên, trung bình nhân là 43,02×. Riêng S6 đạt 1,0357× so với V3. Tại site2/large, S6 giảm từ 28,712 xuống 28,190 ms, Total giảm từ 62,874 xuống 62,197 ms. Các bảng và phép tính bên dưới đọc trực tiếp từ báo cáo P9 đã commit. [Benchmark P9-T05](docs/report/p9-t05-v4-apod.md)

Nghiệm thu P9-T02 đối chiếu sáu cặp synthetic và 12 cấu hình thật. Mười ca ra panorama đều khớp hình học và có Max RGB = 0. Challenge medium và large cùng trả `NumericalFailure` như baseline. Hai target đã chạy memcheck đều báo 0 lỗi và 0 byte rò rỉ. [Nghiệm thu V4](docs/report/p9-v4-verification.md)

#### Phân tích

Trong lượt benchmark này, việc gộp S6 chỉ rút ngắn thời gian một chút. Sau tối ưu, S6 vẫn chiếm khoảng 45,3% Total ở site2/large. Tám trace NSYS xác nhận chuỗi clear, copy, gather đã được thay bằng một renderer duy nhất. Mười hai report NCU cho thấy tổng lưu lượng DRAM đọc và ghi trong phạm vi kernel được lấy mẫu tăng 13,6–26,0%. Phạm vi counter của V3 chưa gồm bước memset canvas, nên chưa xác định được tổng DRAM của cả S6.

Ở ca chính, năm khối đo luân phiên cho median speedup 1,0270×, dao động từ 0,9774 đến 1,0409×, với bốn khối có lợi cho V4. Kết quả này chưa đạt mức 1,05× đề ra lúc đầu. Nhóm giữ V4 như một thử nghiệm gộp kernel đã đạt parity, và báo cáo cả trường hợp chậm hơn lẫn giới hạn của profiling. [Độ ổn định và phân tích profiler](docs/report/p9-t05-v4-apod.md#5-analysis)


In [ ]:
from IPython.display import Markdown, display
import re

P9_BENCHMARK = "docs/report/p9-t05-v4-apod.md"
P9_VERIFICATION = "docs/report/p9-v4-verification.md"
p9_text = (REPO_ROOT / P9_BENCHMARK).read_text(encoding="utf-8")
p9_totals = report_table(P9_BENCHMARK, ("Site", "Size", "V0 ms"))
p9_s6 = report_table(P9_BENCHMARK, ("Site", "Size", "S6 V3 ms"))

def display_report_table(rows):
    columns = list(rows[0])
    lines = ["| " + " | ".join(columns) + " |", "|" + "---|" * len(columns)]
    lines.extend("| " + " | ".join(row[column] for column in columns) + " |" for row in rows)
    display(Markdown("\n".join(lines)))

print("P9 official historical benchmark · Tesla T4 · measured 19/11/2026")
print("Build/source metadata from the committed experiment report:")
print(p9_text.split("## 6. Reproducibility và bàn giao", 1)[1].strip().split(
    "- Gói gốc", 1)[0].strip())
display_report_table(p9_totals)
display_report_table(p9_s6)
p9_total_ratios = [float(row["V3 ms"]) / float(row["V4 ms"]) for row in p9_totals]
p9_v0_ratios = [float(row["V0 ms"]) / float(row["V4 ms"]) for row in p9_totals]
p9_s6_ratios = [float(row["S6 V3 ms"]) / float(row["S6 V4 ms"]) for row in p9_s6]
print(f"Standard cases faster than V3: {sum(r > 1 for r in p9_total_ratios)}/{len(p9_total_ratios)}")
print(f"Geometric mean V4/V3 end-to-end speedup: {geomean(p9_total_ratios):.4f}x")
print(f"Geometric mean V4/V0 end-to-end speedup: {geomean(p9_v0_ratios):.2f}x")
print(f"Geometric mean V4/V3 S6 speedup: {geomean(p9_s6_ratios):.4f}x")
print("Recomputed ratios use the rounded millisecond values in the report tables.")
print("This historical session is distinct from the live comparison in Section 7.")
display(Markdown(p9_text.split("### Độ ổn định tại ca chính", 1)[1].split(
    "### Profiling S6", 1)[0]))


#### Bằng chứng profiling

Các bảng dưới đọc từ báo cáo P9. Hình timeline và biểu đồ DRAM do Đế tạo, lấy từ thư mục `slides/generated`. Với V3, phạm vi NCU gồm kernel chép A và kernel gather B, còn bước memset chỉ thấy được trong NSYS.


In [ ]:
import matplotlib.pyplot as plt

p9_nsys = report_table(P9_BENCHMARK, ("Ca", "V3 clear (ms)", "V3 copy A (ms)"))
p9_ncu = report_table(P9_BENCHMARK, ("Ca", "V3 đọc (MB)", "V4 đọc (MB)"))
print("Profiler measurements from P9 · separate runs from the five-run timing benchmark")
profile_count = re.search(r"\*\*(\d+) trace NSYS và (\d+) report NCU\*\*", p9_text)
if profile_count is None:
    raise ValueError("Profiler coverage statement is missing from the P9 report.")
print(f"Profiler coverage: {profile_count.group(1)} NSYS traces, {profile_count.group(2)} NCU reports.")
display_report_table(p9_nsys)
display_report_table(p9_ncu)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
labels = [row["Site"] + "/" + row["Size"] for row in p9_totals]
axes[0].barh(labels, p9_total_ratios)
axes[0].axvline(1, color="black", linewidth=1)
axes[0].set(xlabel="T(V3) / T(V4)", title="P9 end-to-end speedup")
axes[1].barh(labels, p9_s6_ratios)
axes[1].axvline(1, color="black", linewidth=1)
axes[1].set(xlabel="S6(V3) / S6(V4)", title="P9 S6 speedup")
plt.show()
plt.close(fig)
for relative, caption in (
    ("slides/generated/nsys-v4.png", "NSYS timeline · site2/large"),
    ("slides/generated/ncu-v4.png", "NCU sampled DRAM traffic"),
):
    from IPython.display import Image as ReportImage
    print(caption)
    display(ReportImage(filename=str(REPO_ROOT / relative)))


#### Bằng chứng nghiệm thu

Phần synthetic so tập điểm Q với ground truth. Với các cấu hình ảnh thật, status được kiểm tra trước, sau đó mới đối chiếu H, canvas và RGB ở các ca thành công. Kết quả memcheck chỉ áp dụng cho các target có trong bảng.


In [ ]:
p9_synthetic = report_table(P9_VERIFICATION, ("Ca kiểm thử", "Số điểm Q"))
p9_real = report_table(P9_VERIFICATION, ("Cấu hình", "Cặp ảnh"))
p9_memory = report_table(P9_VERIFICATION, ("Mục tiêu kiểm tra", "Lệnh thực thi"))
print("Historical P9-T02 acceptance · Tesla T4 · source ab8df98")
display_report_table(p9_synthetic)
display_report_table(p9_real)
display_report_table(p9_memory)

def reported_number(value):
    return float(value.removesuffix(" px").replace(",", "."))

p9_success = [row for row in p9_real if row["Trạng thái V0–V4"].startswith("Success")]
print(f"Synthetic cases: {len(p9_synthetic)}")
print(f"Largest synthetic RMSE on Q: {max(reported_number(row['V4 RMSE vs H_gt']) for row in p9_synthetic):.3f} px")
print(f"Largest synthetic point error on Q: {max(reported_number(row['V4 Max vs H_gt']) for row in p9_synthetic):.3f} px")
print(f"Real status comparisons: {len(p9_real)}; successful panoramas: {len(p9_success)}")
print("Max RGB lists and exact canvas offsets are printed in the real-image matrix above.")
print("Memcheck coverage consists of the suite and CLI targets shown above.")
show_source("src/v4/s6/v4_warp_blend.cu", ("const bool has_a =", "const double blended =",
                                         "const dim3 block(", "render_canvas_kernel<<<"))


In [ ]:
v4_summary = BENCHMARK_DIR / "v4" / "summary.csv"
if v4_summary.is_file():
    print(f"Current-session V4 table: {v4_summary.relative_to(BENCHMARK_DIR)}")
else:
    print("Current-session V4 timing is unavailable. The P9 historical output above remains identified by its report.")


#### Kết quả V4 trong lượt benchmark chung

Khi bản build nhận `--version 4`, V4 được đo cùng lượt với các phiên bản khác. Ô dưới in thời gian từng stage của V4 và speedup của nó so với V0 và V3.

In [ ]:
if "V4" in benchmark.versions:
    display(Markdown(fn.stage_table(benchmark, "V4")))
    display(Markdown(fn.speedup_table(benchmark, "V4")))
else:
    print("V4 is not part of this benchmark session.")

### V5 · Hai CUDA stream cho hai nhánh ảnh

#### Bottleneck

V4 dùng một stream: `H2D(A) → H2D(B) → S1(A,B) → S2(A,B) → S3(A,B) → S4`. Trong mỗi stage, A được xử lý trước B. Ở S2, host nhận candidate count rồi đồng bộ stream trước khi sort top-K; lời gọi chép count vào vùng host pageable cũng có thể chặn host. Vì vậy việc phát công việc cho B có thể bị trì hoãn dù hai ảnh chưa phụ thuộc nhau trước S4. [Driver](src/main.cu) · [Pipeline V4](src/v4/v4_pipeline.cu) · [S2](src/v1/s2/v1_detect.cu)

Baseline P10-T01 đo đoạn từ H2D đầu tiên đến features ready trước S4 ở ba ca large, kiểm tra VRAM và chọn site2/large trước khi đo V5. Tỷ trọng trong baseline dùng Total của **binary probe có instrumentation**, không phải Total production của phiên benchmark V5. Bảng probe được hiển thị ở ô kết quả dưới. [Baseline P10-T01](docs/report/p10-t01-pres4-baseline.md)

#### Ý tưởng

Cho hai nhánh H2D/S1–S3 chạy trên hai nonblocking stream, dùng scratch, counts, feature arrays và ô pinned nhận candidate count riêng. Khởi tạo pattern BRIEF và trọng số cần dùng trước khi chạy hai nhánh. Control stream chỉ phát S4 khi cả hai nhánh đã hoàn tất; S4, S5 và S6 lần lượt dùng matching V1, homography V3 và renderer V4.

#### Cài đặt

API một ảnh trong `src/v5/v5_image_branch.cu` tách S2 thành phần tìm candidate và phần hoàn tất top-K. Dispatcher phát **prefix** H2D/S1/S2-candidates cho A và B trước khi chờ count. Event `count_ready` đứng sau D2H count vào pinned slot; host dùng `cudaEventQuery` để hỏi trạng thái. Nhánh đã sẵn sàng được phát **suffix** S2-topK/S3 và ghi event `done`. Control stream dùng `cudaStreamWaitEvent` chờ cả hai `done`, chạy `merge_counts` hợp nhất số lượng/trạng thái, ghi event ready rồi phát S4–S6 và D2H panorama. Việc query tránh chặn host tại một event chưa xong; vòng điều phối vẫn có chi phí CPU và gọi `yield` khi chưa nhánh nào hoàn tất.

`--v5-schedule sequential` buộc nhánh B chờ `doneA`, còn `dual` phát cả hai prefix trước. Hai mode dùng cùng API, kernel, workspace và pinned buffers, nên đây là đối chứng để đánh giá lịch chạy. V4 production có cách nhóm stage và scratch khác, do đó không đồng nhất với V5 sequential. [Contract V5](docs/report/p10-v5-contract.md) · [Image branch](src/v5/v5_image_branch.cu) · [Dispatcher](src/v5/v5_pipeline.cu)

#### Kết quả

Ô dưới đọc các bảng P10-T05 chính thức: Total cùng phiên cho V0/V4/V5 sequential/V5 dual, sidecar Pre-S4, năm khối đo luân phiên tại site2/large và chín cấu hình NSYS. Speedup được tính bằng thời gian đối chứng chia thời gian V5 dual. Chín cấu hình chuẩn tạo thành tập tổng hợp; challenge được báo riêng. [Benchmark P10-T05](docs/report/p10-t05-v5-apod.md)

Từ mean đã làm tròn trong bảng, trung bình nhân V5/V0 là **49,876178×**, V5/V4 là **1,068633×**, còn dual/sequential là **1,036413×**. Tại site2/large, Pre-S4 sequential giảm từ **13,700** xuống **10,971 ms**; trung bình nhân tăng tốc Pre-S4 trên chín ca là **1,256430×**. Báo cáo ghi **1,2565×**; cần sidecar gốc để đối soát chữ số cuối. Các giá trị tính lại và nguồn báo cáo được in trong output, không lấy cột speedup đã làm tròn làm đầu vào.

Năm khối đo luân phiên có median V5/V4 **1,066629×**, nhưng chỉ **3/5 khối** có lợi, chưa đạt điều kiện **ít nhất 4/5**. Bảng NSYS ghi overlap V5 dual **16,6–24,5%** và peak VRAM tăng **64,30 MiB** so với V4 tại ba ca large. Đây là kết quả suy ra từ bảng báo cáo đã commit. CSV, sidecar và SQLite gốc được báo cáo định danh bằng SHA-256, nhưng chưa được kiểm độc lập trong notebook này. [Nguồn và môi trường P10-T05](docs/report/p10-t05-v5-apod.md#7-reproducibility-và-bàn-giao)

Baseline T01 vẫn được hiển thị riêng: tỷ trọng pre-S4 thuộc **binary probe có instrumentation** và dùng Total của chính probe, không phải tỷ trọng production. CUDA-event Pre-S4, thời gian host chờ count và interval NSYS có phạm vi khác nhau; không cộng hoặc trừ chéo phiên. Kết quả synthetic/status/canvas/RGB đọc từ nghiệm thu T04, tách khỏi benchmark. [Baseline T01](docs/report/p10-t01-pres4-baseline.md) · [Nghiệm thu V5](docs/report/p10-v5-verification.md)

#### Phân tích

Hai stream tạo overlap thực theo bảng NSYS và giảm elapsed Pre-S4. Lợi ích toàn pipeline nhỏ hơn lợi ích ở đoạn đầu vì S4–S6, D2H và chi phí điều phối vẫn nằm trong Total. Đối chứng sequential/dual dùng cùng API, kernel và bộ nhớ giúp đánh giá lịch chạy; so V5/V4 còn chịu khác biệt cách nhóm stage và workspace. Tổng duration các stage/kernel chồng lấn không phải elapsed pipeline và không dùng làm tỷ trọng Amdahl.

Bảng mean chính có **8/9 ca** nhanh hơn V4; site3/small chậm hơn trong lượt này. Báo cáo có lượt đo lại cho ca đó, nhưng bảng tóm tắt chưa đủ để quy nguyên nhân biến thiên cho xung nhịp hay tranh chấp SM. Tương tự, overlap trong timeline không tự chứng minh occupancy hoặc mức bão hòa tài nguyên. Dù median ca chính đạt ngưỡng, điều kiện nhất quán giữa các khối chưa đạt.

Nhóm giữ **V4 là production baseline**, **V5 là concurrency case study** theo tiêu chí đã chốt trước đo. V5 có lợi trong nhiều phép đo và minh họa đồng thời ở cấp pipeline; số đo hiện tại chưa chứng minh cải thiện end-to-end ổn định. Mục 7 đo lại mọi phiên bản trong cùng phiên Colab, tách khỏi các bảng lịch sử P9/P10.


In [ ]:
from v5_report import analyze, BENCHMARK, PRES4, PROFILE, clean

P10_BENCHMARK = "docs/report/p10-t05-v5-apod.md"
P10_VERIFICATION = "docs/report/p10-v5-verification.md"
p10 = analyze(REPO_ROOT / P10_BENCHMARK, root=REPO_ROOT)
p10_metrics = p10["metrics"]
p10_synthetic = report_table(P10_VERIFICATION, ("Ca kiểm thử", "Số điểm Q", "V5 RMSE vs H_gt"))
p10_real = report_table(P10_VERIFICATION, ("Cấu hình", "Cặp ảnh", "Trạng thái V0/V4/V5"))

def fmt(value, digits=3):
    return f"{value:.{digits}f}"

display(Markdown("#### P10-T05: thời gian cùng phiên Tesla T4 (ms)"))
print("Nguồn: bảng báo cáo đã commit; các mean được làm tròn. Không ghép với phiên P9 hoặc T01.")
display_report_table([{"Ca": "/".join(case), "V0": fmt(r[BENCHMARK[2]]),
                       "V4": fmt(r[BENCHMARK[3]]), "V5 seq": fmt(r[BENCHMARK[4]]),
                       "V5 dual": fmt(r[BENCHMARK[5]])} for case, r in p10["benchmark"].items()])
display_report_table([{"Ca": "/".join(case), "V5/V0": fmt(r[BENCHMARK[2]] / r[BENCHMARK[5]], 4),
                       "V5/V4": fmt(r[BENCHMARK[3]] / r[BENCHMARK[5]], 4),
                       "Dual/Seq": fmt(r[BENCHMARK[4]] / r[BENCHMARK[5]], 4)}
                      for case, r in p10["benchmark"].items()])
print("Speedup = thời gian đối chứng / thời gian V5 dual; không dùng cột speedup đã làm tròn làm input.")
metric_labels = {"geomean_v5_v0": "GM V5/V0 (9 ca)", "geomean_nine": "GM V5/V4 (9 ca)",
                 "geomean_dual_seq": "GM Dual/Seq (9 ca)", "geomean_medium_large": "GM V5/V4 (6 ca medium/large)",
                 "arithmetic_mean": "Trung bình cộng V5/V4", "ratio_of_sums": "Tỉ số tổng mean V4/V5",
                 "primary_case": "V5/V4 tại site2/large", "geomean_pres4": "GM Pre-S4 Seq/Dual (9 ca)"}
display_report_table([{"Thống kê": label, "Tính từ bảng (×)": fmt(p10_metrics[key], 6)}
                      for key, label in metric_labels.items()])
print(f"Nhanh hơn V4 theo bảng mean chính: {p10_metrics['faster_cases']}/9 ca chuẩn.")
for case, r in p10["challenge"].items():
    print(f"Challenge riêng: {'/'.join(case)}, V5/V4 = {r[BENCHMARK[3]] / r[BENCHMARK[5]]:.6f}×.")
print("Challenge medium/large: NumericalFailure theo bảng nghiệm thu bên dưới; không tính speedup.")

display(Markdown("#### Pre-S4: đối chứng lịch chạy và thời gian host"))
display_report_table([{"Ca": "/".join(case), "Seq ms": fmt(r[PRES4[2]]), "Dual ms": fmt(r[PRES4[3]]),
                       "Seq/Dual": fmt(r[PRES4[2]] / r[PRES4[3]], 4), "Host ms": fmt(r[PRES4[4]]),
                       "Wait A ms": fmt(r[PRES4[5]]), "Wait B ms": fmt(r[PRES4[6]])}
                      for case, r in p10["pres4"].items()])
primary = p10["pres4"][("site2", "large")]
seq, dual = primary[PRES4[2]], primary[PRES4[3]]
print(f"site2/large: {seq:.3f} → {dual:.3f} ms; {seq / dual:.6f}×; giảm latency {(1 - dual / seq) * 100:.3f}%.")
print("Pre-S4 dùng CUDA events; Host/Wait là thời gian host riêng. Không cộng các khoảng này.")
print(f"GM từ mean đã làm tròn = {p10_metrics['geomean_pres4']:.6f}×; báo cáo ghi 1,2565×. Cần sidecar gốc để đối soát chữ số cuối.")

# Preserve the separate instrumented T01 session and its explicit denominator.
P10_BASELINE = "docs/report/p10-t01-pres4-baseline.md"
p10_probe = report_table(P10_BASELINE, ("Ca", "CUDA mean", "Median", "Min"))
print("T01 V4 instrumented probe: share = mean(PreS4_i / Total_probe_i), không phải production share.")
display_report_table(p10_probe)

display(Markdown("#### Độ ổn định: 5 khối luân phiên tại site2/large"))
display_report_table([{"Khối": str(block), "V5/V4 tính lại": fmt(ratio, 6),
                       "Có lợi cho V5": "Có" if ratio > 1 else "Không"}
                      for block, ratio in p10["paired"].items()])
print(f"Median tỉ số từng khối = {p10_metrics['paired_median']:.6f}×; có lợi {p10_metrics['beneficial_blocks']}/5 khối.")
print("Median ≥ 1,05×: đạt; số khối có lợi ≥ 4/5: không đạt. Không thay median bằng GM của 9 ca.")

display(Markdown("#### NSYS: timeline, overlap và peak VRAM"))
display_report_table([{"Ca": r[clean(PROFILE[0])], "Mode": r[PROFILE[1]],
                       "Đến S4 ms": r[clean(PROFILE[2])], "Overlap ms": r[PROFILE[3]],
                       "Overlap %": r[PROFILE[4]], "Streams": r[PROFILE[5]],
                       "Peak MiB": r[PROFILE[6]]} for r in p10["profile_rows"]])
print(f"Overlap V5 dual theo NSYS: {p10_metrics['overlap_min']:.1f}%–{p10_metrics['overlap_max']:.1f}%.")
print("Peak VRAM dual trừ V4 tại ba ca large (MiB):", [fmt(x, 2) for x in p10_metrics["vram_increases_mib"]])
print("NSYS và CUDA-event sidecar là các lượt đo riêng. Không lấy latency có profiler làm benchmark chính.")
print("Cột remaining allocations = 0 B trong report không phải VRAM vật lý còn trống hoặc kết quả memcheck.")

provenance = p10["provenance"]
display(Markdown("#### Nguồn bằng chứng và quyết định APOD"))
for key in ("measurement_source_sha", "measured_at", "environment", "report_sha256",
            "archive_reference", "archive_reported_sha256", "raw_archive_present", "raw_independently_verified"):
    print(f"{key}: {provenance[key]}")
print("Các số trên tính từ bảng báo cáo đã commit; CSV/SQLite gốc chưa được kiểm độc lập trong notebook này.")
print("Quyết định chính thức: V4 production baseline; V5 concurrency case study.")
print("Các ngưỡng ổn định giữ nguyên trước đo:", p10["decision"]["thresholds"])

p10_success = [row for row in p10_real if row["Trạng thái V0/V4/V5"].startswith("Success")]
print(f"P10-T04: {len(p10_synthetic)} synthetic, {len(p10_real)} cấu hình thật, {len(p10_success)} panorama thành công.")
print("Max RGB vs V4 trên các ca thành công:", max(int(row["Max RGB vs V4"]) for row in p10_success))
display_report_table(p10_synthetic)
display_report_table(p10_real)


In [ ]:
if "V5" in benchmark.versions:
    display(Markdown(fn.stage_table(benchmark, "V5")))
    display(Markdown(fn.speedup_table(benchmark, "V5")))
else:
    print("V5 is not part of this benchmark session.")

<a id="final-comparison"></a>

## 7. So sánh tổng hợp

Với phiên bản $v$, speedup end-to-end so với V0 là $T_0/T_v$ và speedup so với phiên bản liền trước là $T_{v-1}/T_v$. Ở đây $T$ là thời gian trung bình trên cùng cặp ảnh và cùng độ phân giải, đo trong một phiên Colab. Thời gian chép dữ liệu và tổng thời gian kernel được báo riêng. Sáu khoảng thời gian stage được trình bày cho phiên bản tốt nhất, chọn theo quy tắc nêu ở dưới. Các ca challenge thất bại có status và log, không tính speedup.
Các bảng ở mục này lấy từ cùng BENCHMARK_DIR của phiên hiện tại; bảng P9/P10 trước đó là lịch sử riêng. V5 dùng mode dual mặc định. Các khoảng S1–S3 và H2D của V5 cộng duration hai nhánh, có thể chồng lấn; biểu đồ stage không phải phép phân chia các phần độc lập của elapsed Total. Phiên bản tốt nhất theo geomean của bảng mới được dùng để vẽ, còn quyết định production của nhóm vẫn là V4.


Bảng P10-T05 ở mục 6 phân biệt ba đối chứng: V5/V0, V5/V4 production và V5 dual/sequential. Median năm khối luân phiên dùng để xét độ ổn định, không thay bằng trung bình nhân của chín mean. Phiên đo mới ở đây không thay thế lịch sử các khối hoặc profiling; nếu thứ hạng khác, giữ cả hai nguồn và giải thích theo phiên đo thay vì trộn số.


In [ ]:
env = benchmark.environment
print(f"GPU: {env['gpu']} | CPU: {env['cpu']} | measured at {env['time_utc']} | timed runs: {env['runs']}")
display(Markdown(fn.speedup_table(benchmark)))
display(Markdown(fn.geomean_table(benchmark)))
display(Markdown(fn.failure_table(benchmark)))

Biểu đồ đầu tiên là speedup end-to-end so với V0, nhóm theo site và độ phân giải. Mỗi biểu đồ sau đó tách speedup theo stage cho một phiên bản GPU. Phiên bản tốt nhất là phiên bản có trung bình nhân speedup so với V0 lớn nhất trên các cấu hình chuẩn. Bảng và biểu đồ cột chồng cuối mục cho thấy thời gian từng stage của phiên bản đó.

In [ ]:
for chart in fn.plot_speedup_charts(benchmark, CHART_DIR):
    display(Image(filename=str(chart)))
best_version = fn.best_version(benchmark)
print(f"Best version (geometric-mean speedup vs V0 over the standard configurations): {best_version}")
display(Markdown(fn.stage_table(benchmark, best_version)))
display(Image(filename=str(fn.plot_stage_breakdown(benchmark, best_version, CHART_DIR))))

<a id="results-gallery"></a>

## 8. Kết quả ghép ảnh

Mục này đặt cặp ảnh đầu vào cạnh panorama của các site chuẩn do nhóm chụp, kèm ca challenge với hai ảnh nguồn và panorama bị méo hoặc trạng thái lỗi của nó.

### Phân tích ca challenge thất bại

Trong P8, ở độ phân giải small pipeline vẫn ra panorama, nhưng lan can bị nhân đôi trong vùng overlap và ảnh B bị méo phối cảnh mạnh. Ở medium và large, S6 trả `NumericalFailure`. Log cho biết stage nào báo lỗi. Muốn biết phép kiểm tra hình học cụ thể nào trong S6 thất bại thì cần thêm thông tin chẩn đoán. Các phiên bản cho ra kết quả giống nhau, nên chỗ lệch nhìn thấy được vẫn còn nguyên ở mọi phiên bản. Một phép biến đổi phối cảnh duy nhất cộng với blend trung bình đều có thể để lại sai số ở cảnh có nhiều cấu trúc lặp và thiếu sáng. [Ghi nhận ca challenge trong P8](docs/report/p8-v3-verification.md#7-challenge-observations)

#### Ảnh đầu vào và panorama

Mỗi hàng gồm ảnh A, ảnh B và panorama do phiên bản tốt nhất ở mục 7 tạo ra trong lượt benchmark. Với cấu hình bị lỗi, ô panorama hiện mã lỗi và vài dòng cuối của log. Hàng cuối là cặp challenge ở độ phân giải small, nơi pipeline vẫn ra ảnh nhưng ảnh bị méo. Ảnh mẫu do môn học cung cấp chưa có trong repo nên chưa có mặt trong gallery.

In [ ]:
gallery = fn.gallery_entries(benchmark, DATASET_DIR, best_version, "medium")
gallery += fn.gallery_entries(benchmark, DATASET_DIR, best_version, "small", sites=("challenge",))
display(Image(filename=str(fn.plot_gallery(gallery, CHART_DIR / f"gallery_{best_version.lower()}.jpg"))))
for entry in gallery:
    print(f"{entry['site']}/{entry['size']} {entry['version']}: {entry['status']}")

In [ ]:
print("Historical P8 challenge outcomes")
for row in p8_real:
    if row["Cấu hình"].startswith("challenge/"):
        print(f"{row['Cấu hình']}: {row[p8_real_status]}")

<a id="conclusion"></a>

## 9. Kết luận

Bottleneck thay đổi theo phiên bản và kích thước bài toán. Ở V0, so khớp descriptor chi phối các cỡ nhỏ, còn phát hiện đặc trưng hoặc dựng canvas có thể chi phối ở cỡ lớn. Sau khi đưa pipeline lên CUDA, S5 trở thành phần lớn thời gian V1; V2 và V3 giảm chi phí xử lý giả thuyết, chấm điểm và refit. V4 tiếp tục gộp các lượt dựng canvas S6, giữ phương pháp nội suy/blend và đạt parity trên các ca đã thử, với lợi ích timing nhỏ trong benchmark P9. Counter DRAM của các kernel được lấy mẫu tăng và phạm vi V3 chưa gồm memset, nên giảm số lượt ghi ở source chưa chứng minh lưu lượng DRAM toàn S6 giảm. [APOD V4](docs/report/p9-t05-v4-apod.md)

V5 đưa hai nhánh H2D/S1–S3 lên hai stream rồi hội tụ trước S4, kèm mode sequential cùng kernel/bộ nhớ để đối chứng. Các khoảng pre-S4 trích từ báo cáo cho thấy lợi ích ở đoạn đầu, nhưng mức tăng tốc end-to-end còn phụ thuộc downstream, D2H và điều phối. Tính lại từ bảng P10 cho kết quả khác aggregate trong kết luận APOD, và các ca small ảnh hưởng rõ tới geomean. Thiếu raw CSV, sidecar và trace V5 nên chưa xác minh độc lập độ ổn định, mức overlap hoặc tiêu chí các khối đo ghép cặp. Các số lịch sử được giữ theo đúng phạm vi nguồn; bảng mục 7 là phép so sánh mới trong cùng phiên Colab. [APOD V5](docs/report/p10-t05-v5-apod.md)

Nhóm giữ **V4 làm production baseline** và **V5 làm concurrency case study**. Qua thử nghiệm này, tối ưu lịch chạy cần được đánh giá bằng elapsed Total sau D2H và ablation cùng phiên, bên cạnh timeline; sum duration các kernel chồng lấn không cho speedup end-to-end. Bước đánh giá tiếp theo là thu per-run sidecar và trace gắn cùng SHA/source để tách chi phí chờ count, query host và contention.

Các lượt hồi quy xác nhận sự nhất quán trên fixture/dataset đã thử, trong phạm vi ngưỡng kiểm chứng. Thị sai, overlap hẹp, cấu trúc lặp, vật thể chuyển động và chênh sáng vẫn có thể làm homography hoặc đường nối kém. Pixel khớp giữa các phiên bản không bảo đảm panorama căn chỉnh tốt; ca challenge cần được đọc cùng match, hình học và ảnh đầu ra. [Nghiệm thu V5](docs/report/p10-v5-verification.md)


<a id="references"></a>

## 10. Tài liệu tham khảo

### Phương pháp và cài đặt

- ITU-R. *Recommendation BT.601*, hệ số luma dùng ở S1. [Khuyến nghị](https://www.itu.int/rec/R-REC-BT.601)
- Rosten, E., và Drummond, T. (2006). *Machine Learning for High-Speed Corner Detection*. ECCV. [PDF của tác giả](https://www.edwardrosten.com/work/rosten_2006_machine.pdf)
- Calonder, M., Lepetit, V., Strecha, C., và Fua, P. (2010). *BRIEF: Binary Robust Independent Elementary Features*. ECCV. [Bài báo](https://www.cs.ubc.ca/~lowe/525/papers/calonder_eccv10.pdf)
- Lowe, D. G. (2004). *Distinctive Image Features from Scale-Invariant Keypoints*. IJCV. Cơ sở của ratio test ở S4. [PDF của tác giả](https://www.cs.ubc.ca/~lowe/papers/ijcv04.pdf)
- Brown, M., và Lowe, D. G. (2007). *Automatic Panoramic Image Stitching using Invariant Features*. IJCV. [PDF của tác giả](https://www.cs.ubc.ca/~lowe/papers/07brown.pdf)
- Szeliski, R. (2022). *Computer Vision: Algorithms and Applications*, ấn bản thứ hai. Căn chỉnh ảnh, lấy mẫu lại và ghép ảnh. [Trang của tác giả](https://szeliski.org/Book/)
- Vigna, S. *SplitMix64 reference implementation*, tham chiếu cho hàm trộn của bộ lấy mẫu S5. [Mã tham chiếu](https://prng.di.unimi.it/splitmix64.c)
- NVIDIA. *CUDA C++ Best Practices Guide* và *Nsight Systems User Guide*. [Hướng dẫn CUDA](https://docs.nvidia.com/cuda/archive/13.0.0/cuda-c-best-practices-guide/index.html) · [Nsight Systems](https://docs.nvidia.com/nsight-systems/UserGuide/index.html)
- Barrett, S., và cộng sự. `stb_image` và `stb_image_write`, header đọc ghi ảnh đi kèm repo. [Repo stb](https://github.com/nothings/stb) · [Nguồn tham khảo của từng stage](docs/report/references.md)

### Bằng chứng và khả năng tái lập

- [Đề bài](docs/specs/CSC14120_2026_Final%20Project.md) · [Build và CLI](README.md) · [Dataset chính thức](docs/dataset.md) · [Cấu trúc thư mục kết quả](docs/results-guide.md)
- [Benchmark V2 (P7)](docs/report/p7-t03-v2-apod.md) · [Nghiệm thu V2 (P7)](docs/report/p7-v2-verification.md) · [Benchmark V3 (P8)](docs/report/p8-t03-v3-apod.md) · [Nghiệm thu V3 (P8)](docs/report/p8-v3-verification.md)
- [Benchmark và profiling V4 (P9)](docs/report/p9-t05-v4-apod.md) · [Nghiệm thu V4 (P9)](docs/report/p9-v4-verification.md)
- [Baseline trước S4 (P10-T01)](docs/report/p10-t01-pres4-baseline.md) · [Contract V5](docs/report/p10-v5-contract.md) · [Benchmark V5 (P10-T05)](docs/report/p10-t05-v5-apod.md) · [Nghiệm thu V5](docs/report/p10-v5-verification.md)
- [Hồi quy và đóng băng mã nguồn (P11-T02)](docs/report/p11-t02-freeze-verification.md) · [Lượt chạy nộp bài và các phần chờ tích hợp](docs/report/p9-final-notebook-handoff.md)

Chạy notebook theo thứ tự, từ ô setup source và môi trường tới phần so sánh tổng hợp và gallery. Lượt chạy cuối trên Colab sẽ giữ output của các ô, SHA của source, manifest của dataset, thông tin GPU và toolkit, cùng nguồn gốc của số đo benchmark. Bằng chứng P7, P8, P9 và P10 trong notebook vẫn gắn với phiên đo gốc của chúng.

### Sử dụng công cụ AI

OpenAI Codex hỗ trợ soạn và sắp xếp notebook, đối chiếu mô tả kỹ thuật với code và báo cáo trong repo, và kiểm tra cấu trúc notebook cùng các ô chạy được. Số đo lấy từ các báo cáo thí nghiệm được trích dẫn. Nhóm sẽ tổng hợp thêm việc dùng AI của các thành viên khác trong bản nộp.
